In [1]:
# =========================================================
#  KAGGLE TRAINING PIPELINE (High RAM + GPU Optimization)
# =========================================================
import os
import glob
import json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
BATCH_SIZE = 512  # Large batch size for speed (Kaggle has RAM for this)
IMG_H, IMG_W = 192, 64
MODEL_SAVE_PATH = "/kaggle/working/error_bar_model.h5" # Save to working dir

# --- 2. AUTO-DISCOVER DATA PATHS ---
# Kaggle puts files in /kaggle/input/..., but the subfolders can vary.
# We search for the 'labels' folder to find the root.
print("🔍 Searching for dataset...")
possible_label_dirs = glob.glob("/kaggle/input/**/labels", recursive=True)

if not possible_label_dirs:
    raise FileNotFoundError("Could not find a 'labels' folder. Did you upload the dataset correctly?")

lbl_dir = possible_label_dirs[0] # Take the first match
root_dir = os.path.dirname(lbl_dir)
img_dir = os.path.join(root_dir, "images")

print(f"✅ Found Data Root: {root_dir}")
print(f"   Images: {img_dir}")
print(f"   Labels: {lbl_dir}")

# --- 3. BUILD INDEX (METADATA) ---
json_files = glob.glob(os.path.join(lbl_dir, "*.json"))
print(f"📄 Scanning {len(json_files)} label files...")

paths = []
coords = []  # [x, y]
targets = [] # [top, bot]

for idx, j_file in enumerate(json_files):
    filename = os.path.basename(j_file)
    img_path = os.path.join(img_dir, filename.replace(".json", ".png"))
    
    if not os.path.exists(img_path): continue
    
    with open(j_file, 'r') as f:
        data = json.load(f)
    
    for series in data:
        if "Layout_Markers" in series['label']['lineName']: continue
        for pt in series['points']:
            norm_top = min(1.0, pt['topBarPixelDistance'] / (IMG_H / 2))
            norm_bot = min(1.0, pt['bottomBarPixelDistance'] / (IMG_H / 2))
            
            paths.append(img_path)
            coords.append([int(pt['x']), int(pt['y'])])
            targets.append([norm_top, norm_bot])

print(f"📊 Total Training Patches: {len(paths)}")

# Split
train_paths, val_paths, train_coords, val_coords, train_targs, val_targs = train_test_split(
    paths, coords, targets, test_size=0.15, random_state=42
)

# --- 4. DATA PIPELINE (RAM CACHING) ---
def load_raw_uint8(path, coord, target):
    """Loads image and crops it. Returns uint8 to save RAM."""
    file_bytes = tf.io.read_file(path)
    img = tf.io.decode_png(file_bytes, channels=3)
    
    # Geometry
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    px, py = coord[0], coord[1]
    
    # Pad & Crop
    img_padded = tf.image.pad_to_bounding_box(img, IMG_H, IMG_W, h + 2*IMG_H, w + 2*IMG_W)
    center_x = px + IMG_W
    center_y = py + IMG_H
    crop = tf.image.crop_to_bounding_box(
        img_padded, 
        center_y - (IMG_H // 2), 
        center_x - (IMG_W // 2), 
        IMG_H, IMG_W
    )
    return crop, target

def normalize_op(crop, target):
    """Converts uint8 to float32 on the fly."""
    return tf.cast(crop, tf.float32) / 255.0, target

def create_dataset(p, c, t, is_train=True):
    ds = tf.data.Dataset.from_tensor_slices((p, c, t))
    if is_train: ds = ds.shuffle(len(p))
    
    # 1. Load (CPU Heavy)
    ds = ds.map(load_raw_uint8, num_parallel_calls=tf.data.AUTOTUNE)
    
    # 2. CACHE (Stores uint8 in RAM - Kaggle has 30GB, so this is safe!)
    ds = ds.cache()
    
    # 3. Normalize (GPU/Fast)
    ds = ds.map(normalize_op, num_parallel_calls=tf.data.AUTOTUNE)
    
    # 4. Batch & Prefetch
    if is_train: ds = ds.shuffle(5000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

print("⚙️ Building Pipelines...")
train_ds = create_dataset(train_paths, train_coords, train_targs, is_train=True)
val_ds = create_dataset(val_paths, val_coords, val_targs, is_train=False)

# --- 5. MODEL ARCHITECTURE ---
def build_model():
    model = models.Sequential([
        layers.Input(shape=(IMG_H, IMG_W, 3)),
        
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(2, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

# --- 6. TRAINING ---
model = build_model()
model.summary()

checkpoint = callbacks.ModelCheckpoint(MODEL_SAVE_PATH, save_best_only=True, monitor='val_mae', mode='min', verbose=1)
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

print("🚀 Starting Training...")
print("Note: Epoch 1 will take 10-15 mins to fill RAM cache.")
print("      Epoch 2+ will take seconds.")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint, early_stop]
)

print(f"✅ DONE! Model saved to {MODEL_SAVE_PATH}")

2026-01-30 15:45:10.910144: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769787911.149491      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769787911.217449      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769787911.794431      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769787911.794482      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769787911.794488      55 computation_placer.cc:177] computation placer alr

🔍 Searching for dataset...
✅ Found Data Root: /kaggle/input/error-bar-detection
   Images: /kaggle/input/error-bar-detection/images
   Labels: /kaggle/input/error-bar-detection/labels
📄 Scanning 3000 label files...
📊 Total Training Patches: 197952
⚙️ Building Pipelines...


I0000 00:00:1769787944.247086      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769787944.253182      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 192, 64, 32)    │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 96, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 96, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 48, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 24, 8, 128)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 8, 256)     │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 12, 4, 256)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12288)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     3,145,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - loss: 0.1217 - mae: 0.1743

2026-01-30 16:04:35.318311: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng12{k11=2} for conv %cudnn-conv-bias-activation.14 = (f32[512,128,48,16]{3,2,1,0}, u8[0]{0}) custom-call(f32[512,64,48,16]{3,2,1,0} %bitcast.384, f32[128,64,3,3]{3,2,1,0} %bitcast.391, f32[128]{0} %bitcast.393), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", metadata={op_type="Conv2D" op_name="sequential_1/conv2d_2_1/convolution" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kRelu","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-01-30 16:04:35.541630: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.223391356s
Trying algorithm e


Epoch 1: val_mae improved from inf to 0.04429, saving model to /kaggle/working/error_bar_model.h5


329/329 ━━━━━━━━━━━━━━━━━━━━ 1312s 4s/step - loss: 0.1214 - mae: 0.1741 - val_loss: 0.0053 - val_mae: 0.0443
Epoch 2/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step - loss: 0.0053 - mae: 0.0433
Epoch 2: val_mae did not improve from 0.04429
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 246ms/step - loss: 0.0053 - mae: 0.0433 - val_loss: 0.0056 - val_mae: 0.0469
Epoch 3/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 0.0035 - mae: 0.0344
Epoch 3: val_mae did not improve from 0.04429
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 243ms/step - loss: 0.0035 - mae: 0.0344 - val_loss: 0.0056 - val_mae: 0.0447
Epoch 4/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - loss: 0.0028 - mae: 0.0299
Epoch 4: val_mae did not improve from 0.04429
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 244ms/step - loss: 0.0028 - mae: 0.0299 - val_loss: 0.0064 - val_mae: 0.0476
Epoch 5/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 0.0024 - mae: 0.0274
Epoch 5: val_mae improved from 0.04429 to 0.04000, saving model to /kaggle/working/error_ba

329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 243ms/step - loss: 0.0024 - mae: 0.0274 - val_loss: 0.0046 - val_mae: 0.0400
Epoch 6/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - loss: 0.0020 - mae: 0.0250
Epoch 6: val_mae did not improve from 0.04000
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 244ms/step - loss: 0.0020 - mae: 0.0250 - val_loss: 0.0046 - val_mae: 0.0403
Epoch 7/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 233ms/step - loss: 0.0018 - mae: 0.0238
Epoch 7: val_mae did not improve from 0.04000
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 243ms/step - loss: 0.0018 - mae: 0.0238 - val_loss: 0.0051 - val_mae: 0.0422
Epoch 8/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - loss: 0.0016 - mae: 0.0222
Epoch 8: val_mae did not improve from 0.04000
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 244ms/step - loss: 0.0016 - mae: 0.0222 - val_loss: 0.0046 - val_mae: 0.0421
Epoch 9/20
329/329 ━━━━━━━━━━━━━━━━━━━━ 0s 234ms/step - loss: 0.0016 - mae: 0.0219
Epoch 9: val_mae did not improve from 0.04000
329/329 ━━━━━━━━━━━━━━━━━━━━ 81s 244ms/step 